### Imports

In [48]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from ase.io import read
import json
import pickle

# Task 4

### Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

INVENTORY_CSV = OUTPUT_ROOT / "dataset_inventory.csv"
MASTER_CSV = OUTPUT_ROOT / "dataset_master.csv"

PARSED_DIR = OUTPUT_ROOT / "parsed_structures"
ATOM_TABLE_DIR = PARSED_DIR / "atom_tables"

STRUCTURE_METADATA_CSV = PARSED_DIR / "structure_metadata.csv"
FAILED_PARSE_CSV = PARSED_DIR / "failed_parses.csv"

SAVE_ATOM_TABLES = True

MAX_STRUCTURES = None

### Output Directories

In [3]:
PARSED_DIR.mkdir(parents=True, exist_ok=True)
ATOM_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

Project root: D:\masters_project
Data root: D:\masters_project\data
Output root: D:\masters_project\outputs


### Load Dataset Inventories

In [4]:
df_inventory = pd.read_csv(INVENTORY_CSV)

if MAX_STRUCTURES is not None:
    df_inventory = df_inventory.head(MAX_STRUCTURES)

print("Structures to process:", len(df_inventory))

df_inventory.head()

Structures to process: 2916


,structure_id,relative_cif_path,lower_rotation,displacement,upper_rotation
0,L0_D0_U0,r0/t0/t0_0.cif,0.0,0.0,0.0
1,L0_D0_U20,r0/t0/t0_20.cif,0.0,0.0,20.0
2,L0_D0_U40,r0/t0/t0_40.cif,0.0,0.0,40.0
3,L0_D0_U60,r0/t0/t0_60.cif,0.0,0.0,60.0
4,L0_D0_U80,r0/t0/t0_80.cif,0.0,0.0,80.0


### Helper

In [ ]:
def resolve_cif_path(relative_path: str, data_root: Path) -> Path:
    return data_root / relative_path

def read_structure(cif_path: Path):
    try:
        atoms = read(str(cif_path))

        if atoms is None or len(atoms) == 0:
            return None, "empty_structure", "No atoms found"

        return atoms, "ok", None

    except Exception as e:
        return None, "parse_error", f"{type(e).__name__}: {e}"

def get_atom_labels(atoms):
    labels = None

    if hasattr(atoms, "arrays") and "labels" in atoms.arrays:
        labels = list(atoms.arrays["labels"])

    if labels is None:
        labels = [f"{s}{i+1}" for i, s in enumerate(atoms.get_chemical_symbols())]

    return labels

def build_atom_table(atoms):
    symbols = atoms.get_chemical_symbols()
    atomic_numbers = atoms.get_atomic_numbers()

    cart = atoms.get_positions()
    frac = atoms.get_scaled_positions()

    labels = get_atom_labels(atoms)

    df = pd.DataFrame({

        "atom_index": np.arange(len(atoms)),
        "atom_label": labels,
        "symbol": symbols,
        "atomic_number": atomic_numbers,

        "frac_x": frac[:,0],
        "frac_y": frac[:,1],
        "frac_z": frac[:,2],

        "x": cart[:,0],
        "y": cart[:,1],
        "z": cart[:,2]

    })
    return df

def build_structure_metadata(atoms, structure_id, relative_path):

    cell = atoms.get_cell()
    pbc = atoms.get_pbc()

    symbols = atoms.get_chemical_symbols()
    counts = pd.Series(symbols).value_counts().to_dict()

    metadata = {

        "structure_id": structure_id,
        "relative_cif_path": relative_path,

        "atom_count": len(atoms),

        "cell_a": float(cell.lengths()[0]),
        "cell_b": float(cell.lengths()[1]),
        "cell_c": float(cell.lengths()[2]),

        "cell_alpha": float(cell.angles()[0]),
        "cell_beta": float(cell.angles()[1]),
        "cell_gamma": float(cell.angles()[2]),

        "pbc_x": bool(pbc[0]),
        "pbc_y": bool(pbc[1]),
        "pbc_z": bool(pbc[2]),

        "count_C": counts.get("C",0),
        "count_H": counts.get("H",0),
        "count_O": counts.get("O",0),

        "unique_elements": json.dumps(sorted(set(symbols)))
    }

    return metadata

### Test One Structure

In [6]:
sample = df_inventory.iloc[0]

structure_id = sample["structure_id"]
relative_path = sample["relative_cif_path"]

cif_path = resolve_cif_path(relative_path, DATA_ROOT)

atoms, status, error = read_structure(cif_path)

print("Structure:", structure_id)
print("Status:", status)

if atoms is not None:

    atom_table = build_atom_table(atoms)
    metadata = build_structure_metadata(atoms, structure_id, relative_path)

    atom_table.head()

Structure: L0_D0_U0
Status: ok


### Parsing all files

In [7]:
metadata_rows = []
failed_rows = []

for _, row in tqdm(df_inventory.iterrows(), total=len(df_inventory)):

    structure_id = row["structure_id"]
    relative_path = row["relative_cif_path"]

    cif_path = resolve_cif_path(relative_path, DATA_ROOT)

    atoms, status, error = read_structure(cif_path)

    if status != "ok":

        failed_rows.append({
            "structure_id": structure_id,
            "relative_cif_path": relative_path,
            "status": status,
            "error": error
        })

        continue

    atom_table = build_atom_table(atoms)

    atom_table["structure_id"] = structure_id
    atom_table["relative_cif_path"] = relative_path

    if SAVE_ATOM_TABLES:

        save_path = ATOM_TABLE_DIR / f"{structure_id}.csv"
        atom_table.to_csv(save_path, index=False)

    metadata = build_structure_metadata(atoms, structure_id, relative_path)
    metadata_rows.append(metadata)

  0%|          | 0/2916 [00:00<?, ?it/s]

d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegr

### Saving & Validating Results

In [8]:
df_metadata = pd.DataFrame(metadata_rows)
df_failed = pd.DataFrame(failed_rows)

df_metadata.to_csv(STRUCTURE_METADATA_CSV, index=False)
df_failed.to_csv(FAILED_PARSE_CSV, index=False)

print("Saved metadata:", STRUCTURE_METADATA_CSV)
print("Saved failures:", FAILED_PARSE_CSV)

Saved metadata: D:\masters_project\outputs\parsed_structures\structure_metadata.csv
Saved failures: D:\masters_project\outputs\parsed_structures\failed_parses.csv


In [9]:
print("Total structures:", len(df_inventory))
print("Parsed:", len(df_metadata))
print("Failed:", len(df_failed))

print()

print("Atom counts:", df_metadata["atom_count"].unique())
print("C counts:", df_metadata["count_C"].unique())
print("H counts:", df_metadata["count_H"].unique())
print("O counts:", df_metadata["count_O"].unique())

df_metadata.head()

Total structures: 2916
Parsed: 2916
Failed: 0

Atom counts: [156]
C counts: [48]
H counts: [72]
O counts: [36]


,structure_id,relative_cif_path,atom_count,cell_a,cell_b,cell_c,cell_alpha,cell_beta,cell_gamma,pbc_x,pbc_y,pbc_z,count_C,count_H,count_O,unique_elements
0,L0_D0_U0,r0/t0/t0_0.cif,156,18.699181,17.907993,28.180093,90.028496,94.220950,85.653614,True,True,True,48,72,36,"[""C"", ""H"", ""O""]"
1,L0_D0_U20,r0/t0/t0_20.cif,156,18.711026,18.196940,27.761063,88.624316,94.019045,84.670210,True,True,True,48,72,36,"[""C"", ""H"", ""O""]"
2,L0_D0_U40,r0/t0/t0_40.cif,156,18.741948,17.944287,28.144615,89.981893,94.662722,84.103304,True,True,True,48,72,36,"[""C"", ""H"", ""O""]"
3,L0_D0_U60,r0/t0/t0_60.cif,156,18.703983,17.909626,28.203959,89.952891,94.058016,84.711373,True,True,True,48,72,36,"[""C"", ""H"", ""O""]"
4,L0_D0_U80,r0/t0/t0_80.cif,156,18.734436,18.234442,27.898071,88.434911,97.224172,83.408775,True,True,True,48,72,36,"[""C"", ""H"", ""O""]"


# Task 5

### Paths

In [10]:
LAYERED_DIR = OUTPUT_ROOT / "layered_structures"
LAYERED_ATOM_TABLE_DIR = LAYERED_DIR / "atom_tables"

LAYER_METADATA_CSV = LAYERED_DIR / "layer_metadata.csv"
LAYER_VALIDATION_CSV = LAYERED_DIR / "layer_validation.csv"
LAYER_FAILURES_CSV = LAYERED_DIR / "layer_failures.csv"

MAX_STRUCTURES = None   # set small number like 5 for testing
EXPECTED_ATOM_COUNT = 156
EXPECTED_LOWER_COUNT = 78
EXPECTED_UPPER_COUNT = 78

In [11]:
LAYERED_DIR.mkdir(parents=True, exist_ok=True)
LAYERED_ATOM_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input atom tables:", ATOM_TABLE_DIR)
print("Layered output:", LAYERED_DIR)

Project root: D:\masters_project
Input atom tables: D:\masters_project\outputs\parsed_structures\atom_tables
Layered output: D:\masters_project\outputs\layered_structures


In [12]:
atom_table_files = sorted(ATOM_TABLE_DIR.glob("*.csv"))

if MAX_STRUCTURES is not None:
    atom_table_files = atom_table_files[:MAX_STRUCTURES]

print("Atom tables found:", len(atom_table_files))

Atom tables found: 2916


[WindowsPath('D:/masters_project/outputs/parsed_structures/atom_tables/L0_D0_U0.csv'),
 WindowsPath('D:/masters_project/outputs/parsed_structures/atom_tables/L0_D0_U100.csv'),
 WindowsPath('D:/masters_project/outputs/parsed_structures/atom_tables/L0_D0_U120.csv'),
 WindowsPath('D:/masters_project/outputs/parsed_structures/atom_tables/L0_D0_U140.csv'),
 WindowsPath('D:/masters_project/outputs/parsed_structures/atom_tables/L0_D0_U160.csv')]

### Helpers

In [ ]:
def assign_layers_by_z(df_atoms: pd.DataFrame) -> pd.DataFrame:
    required_cols = {"atom_index", "z"}
    missing = required_cols - set(df_atoms.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if len(df_atoms) != EXPECTED_ATOM_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_ATOM_COUNT} atoms, found {len(df_atoms)}"
        )

    df_sorted = df_atoms.sort_values(
        by=["z", "atom_index"],
        ascending=[True, True]
    ).reset_index(drop=True)

    df_sorted["z_rank"] = np.arange(len(df_sorted), dtype=int)
    df_sorted["layer_label"] = 1
    df_sorted.loc[:EXPECTED_LOWER_COUNT - 1, "layer_label"] = 0
    df_sorted["layer_name"] = df_sorted["layer_label"].map({
        0: "lower",
        1: "upper"
    })

    return df_sorted

def build_layer_validation(df_layered: pd.DataFrame, structure_id: str) -> dict:
    lower_count = int((df_layered["layer_label"] == 0).sum())
    upper_count = int((df_layered["layer_label"] == 1).sum())

    validation = {
        "structure_id": structure_id,
        "atom_count": int(len(df_layered)),
        "lower_count": lower_count,
        "upper_count": upper_count,
        "lower_ok": lower_count == EXPECTED_LOWER_COUNT,
        "upper_ok": upper_count == EXPECTED_UPPER_COUNT,
        "total_ok": len(df_layered) == EXPECTED_ATOM_COUNT,
        "z_min": float(df_layered["z"].min()),
        "z_max": float(df_layered["z"].max()),
        "lower_z_max": float(df_layered.loc[df_layered["layer_label"] == 0, "z"].max()),
        "upper_z_min": float(df_layered.loc[df_layered["layer_label"] == 1, "z"].min()),
    }

    validation["layer_gap"] = validation["upper_z_min"] - validation["lower_z_max"]

    return validation

### Testing one structure

In [17]:
sample_path = atom_table_files[0]
df_sample = pd.read_csv(sample_path)

df_sample_layered = assign_layers_by_z(df_sample)

print("Sample file:", sample_path.name)
display(df_sample_layered.head())
display(df_sample_layered.tail())

sample_structure_id = df_sample_layered["structure_id"].iloc[0]
sample_validation = build_layer_validation(df_sample_layered, sample_structure_id)
sample_validation

Sample file: L0_D0_U0.csv


,atom_index,atom_label,symbol,atomic_number,frac_x,frac_y,frac_z,x,y,z,structure_id,relative_cif_path,z_rank,layer_label,layer_name
0,86,H87,H,1,0.622223,0.317492,0.059558,11.942418,5.677838,1.673773,L0_D0_U0,r0/t0/t0_0.cif,0,0,lower
1,50,H51,H,1,0.122246,0.317540,0.059580,2.593276,5.678713,1.674395,L0_D0_U0,r0/t0/t0_0.cif,1,0,lower
2,54,H55,H,1,0.152765,0.447070,0.077661,3.302254,7.994250,2.182526,L0_D0_U0,r0/t0/t0_0.cif,2,0,lower
3,90,H91,H,1,0.652836,0.447003,0.077757,12.652875,7.993063,2.185238,L0_D0_U0,r0/t0/t0_0.cif,3,0,lower
4,53,H54,H,1,0.056777,0.453083,0.081386,1.507792,8.102162,2.287225,L0_D0_U0,r0/t0/t0_0.cif,4,0,lower


,atom_index,atom_label,symbol,atomic_number,frac_x,frac_y,frac_z,x,y,z,structure_id,relative_cif_path,z_rank,layer_label,layer_name
151,79,H80,H,1,0.834894,0.277133,0.563263,14.819659,5.029504,15.829538,L0_D0_U0,r0/t0/t0_0.cif,151,1,upper
152,137,O138,O,8,0.771277,0.185588,0.565187,13.501846,3.395096,15.883603,L0_D0_U0,r0/t0/t0_0.cif,152,1,upper
153,155,O156,O,8,0.271783,0.185016,0.565278,4.160764,3.384909,15.886162,L0_D0_U0,r0/t0/t0_0.cif,153,1,upper
154,115,H116,H,1,0.377136,0.190735,0.587032,6.093420,3.490155,16.497536,L0_D0_U0,r0/t0/t0_0.cif,154,1,upper
155,80,H81,H,1,0.876530,0.191352,0.587204,15.432153,3.501192,16.502377,L0_D0_U0,r0/t0/t0_0.cif,155,1,upper


{'structure_id': 'L0_D0_U0',
 'atom_count': 156,
 'lower_count': 78,
 'upper_count': 78,
 'lower_ok': True,
 'upper_ok': True,
 'total_ok': True,
 'z_min': 1.6737729567368778,
 'z_max': 16.502377356323755,
 'lower_z_max': 8.488367538689104,
 'upper_z_min': 9.465630861419854,
 'layer_gap': 0.9772633227307495}

### All cif structures

In [18]:
metadata_rows = []
validation_rows = []
failure_rows = []

for file_path in tqdm(atom_table_files, desc="Assigning layers"):
    try:
        df_atoms = pd.read_csv(file_path)

        structure_id = df_atoms["structure_id"].iloc[0]
        relative_cif_path = df_atoms["relative_cif_path"].iloc[0]

        df_layered = assign_layers_by_z(df_atoms)

        save_path = LAYERED_ATOM_TABLE_DIR / file_path.name
        df_layered.to_csv(save_path, index=False)

        metadata_rows.append({
            "structure_id": structure_id,
            "relative_cif_path": relative_cif_path,
            "atom_table_file": file_path.name,
            "layered_atom_table_file": save_path.name
        })

        validation_rows.append(
            build_layer_validation(df_layered, structure_id)
        )

    except Exception as e:
        structure_id = None
        relative_cif_path = None

        try:
            if "df_atoms" in locals() and len(df_atoms) > 0:
                structure_id = df_atoms.get("structure_id", pd.Series([None])).iloc[0]
                relative_cif_path = df_atoms.get("relative_cif_path", pd.Series([None])).iloc[0]
        except Exception:
            pass

        failure_rows.append({
            "file_name": file_path.name,
            "structure_id": structure_id,
            "relative_cif_path": relative_cif_path,
            "error": f"{type(e).__name__}: {e}"
        })

Assigning layers:   0%|          | 0/2916 [00:00<?, ?it/s]

### Saving outputs and validating

In [19]:
df_layer_metadata = pd.DataFrame(metadata_rows)
df_layer_validation = pd.DataFrame(validation_rows)
df_layer_failures = pd.DataFrame(failure_rows)

df_layer_metadata.to_csv(LAYER_METADATA_CSV, index=False)
df_layer_validation.to_csv(LAYER_VALIDATION_CSV, index=False)
df_layer_failures.to_csv(LAYER_FAILURES_CSV, index=False)

print("Saved:", LAYER_METADATA_CSV)
print("Saved:", LAYER_VALIDATION_CSV)
print("Saved:", LAYER_FAILURES_CSV)

Saved: D:\masters_project\outputs\layered_structures\layer_metadata.csv
Saved: D:\masters_project\outputs\layered_structures\layer_validation.csv
Saved: D:\masters_project\outputs\layered_structures\layer_failures.csv


In [20]:
print("=" * 60)
print("LAYER ASSIGNMENT SUMMARY")
print("=" * 60)
print("Requested structures:", len(atom_table_files))
print("Processed successfully:", len(df_layer_validation))
print("Failures:", len(df_layer_failures))

if len(df_layer_validation) > 0:
    print()
    print("All total_ok :", bool(df_layer_validation["total_ok"].all()))
    print("All lower_ok :", bool(df_layer_validation["lower_ok"].all()))
    print("All upper_ok :", bool(df_layer_validation["upper_ok"].all()))

display(df_layer_validation.head())

if len(df_layer_failures) > 0:
    display(df_layer_failures.head())

LAYER ASSIGNMENT SUMMARY
Requested structures: 2916
Processed successfully: 2916
Failures: 0

All total_ok : True
All lower_ok : True
All upper_ok : True


,structure_id,atom_count,lower_count,upper_count,lower_ok,upper_ok,total_ok,z_min,z_max,lower_z_max,upper_z_min,layer_gap
0,L0_D0_U0,156,78,78,True,True,True,1.673773,16.502377,8.488368,9.465631,0.977263
1,L0_D0_U100,156,78,78,True,True,True,1.586208,13.828761,8.430282,9.755171,1.324889
2,L0_D0_U120,156,78,78,True,True,True,1.643392,13.854068,8.465692,9.740601,1.274909
3,L0_D0_U140,156,78,78,True,True,True,1.655027,15.252929,8.491622,9.875301,1.383679
4,L0_D0_U160,156,78,78,True,True,True,1.695572,16.687739,8.510567,10.268464,1.757897


# Task 6

### Paths

In [24]:
FEATURE_DIR = OUTPUT_ROOT / "structural_features"
FEATURE_CSV = FEATURE_DIR / "structure_features.csv"
FEATURE_FAILURES_CSV = FEATURE_DIR / "feature_failures.csv"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Layered atom tables:", LAYERED_ATOM_TABLE_DIR)
print("Feature output:", FEATURE_DIR)

Project root: D:\masters_project
Layered atom tables: D:\masters_project\outputs\layered_structures\atom_tables
Feature output: D:\masters_project\outputs\structural_features


### Discover Input Files

In [25]:
atom_table_files = sorted(LAYERED_ATOM_TABLE_DIR.glob("*.csv"))

if MAX_STRUCTURES is not None:
    atom_table_files = atom_table_files[:MAX_STRUCTURES]

print("Layered atom tables found:", len(atom_table_files))

Layered atom tables found: 2916


[WindowsPath('D:/masters_project/outputs/layered_structures/atom_tables/L0_D0_U0.csv'),
 WindowsPath('D:/masters_project/outputs/layered_structures/atom_tables/L0_D0_U100.csv'),
 WindowsPath('D:/masters_project/outputs/layered_structures/atom_tables/L0_D0_U120.csv'),
 WindowsPath('D:/masters_project/outputs/layered_structures/atom_tables/L0_D0_U140.csv'),
 WindowsPath('D:/masters_project/outputs/layered_structures/atom_tables/L0_D0_U160.csv')]

### Helper

In [ ]:
ATOMIC_MASS = {
    "H": 1.008,
    "C": 12.011,
    "O": 15.999,
}

def validate_layered_table(df: pd.DataFrame) -> None:
    required_cols = {
        "atom_index",
        "symbol",
        "x", "y", "z",
        "layer_label",
        "structure_id",
        "relative_cif_path",
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if len(df) != EXPECTED_ATOM_COUNT:
        raise ValueError(f"Expected {EXPECTED_ATOM_COUNT} atoms, found {len(df)}")

    lower_count = int((df["layer_label"] == 0).sum())
    upper_count = int((df["layer_label"] == 1).sum())

    if lower_count != EXPECTED_LOWER_COUNT:
        raise ValueError(f"Expected {EXPECTED_LOWER_COUNT} lower atoms, found {lower_count}")

    if upper_count != EXPECTED_UPPER_COUNT:
        raise ValueError(f"Expected {EXPECTED_UPPER_COUNT} upper atoms, found {upper_count}")

def get_positions(df: pd.DataFrame) -> np.ndarray:
    return df[["x", "y", "z"]].to_numpy(dtype=float)


def pairwise_distance_matrix(positions_a: np.ndarray, positions_b: np.ndarray | None = None) -> np.ndarray:
    if positions_b is None:
        positions_b = positions_a

    diff = positions_a[:, None, :] - positions_b[None, :, :]
    dist = np.linalg.norm(diff, axis=2)
    return dist

def compute_centroid(df: pd.DataFrame) -> np.ndarray:
    pos = get_positions(df)
    return pos.mean(axis=0)


def compute_center_of_mass(df: pd.DataFrame) -> np.ndarray:
    masses = df["symbol"].map(ATOMIC_MASS).to_numpy(dtype=float)
    pos = get_positions(df)
    return np.average(pos, axis=0, weights=masses)

def summarize_intralayer_distances(df_layer: pd.DataFrame) -> dict:
    pos = get_positions(df_layer)
    dist = pairwise_distance_matrix(pos)
    upper_tri = dist[np.triu_indices(len(df_layer), k=1)]

    return {
        "count": int(len(upper_tri)),
        "mean": float(np.mean(upper_tri)),
        "std": float(np.std(upper_tri)),
        "min": float(np.min(upper_tri)),
        "max": float(np.max(upper_tri)),
    }


def summarize_interlayer_distances(df_lower: pd.DataFrame, df_upper: pd.DataFrame) -> dict:
    pos_lower = get_positions(df_lower)
    pos_upper = get_positions(df_upper)

    dist = pairwise_distance_matrix(pos_lower, pos_upper)
    flat = dist.ravel()

    return {
        "count": int(len(flat)),
        "mean": float(np.mean(flat)),
        "std": float(np.std(flat)),
        "min": float(np.min(flat)),
        "max": float(np.max(flat)),
    }
    
def summarize_nearest_neighbors(df: pd.DataFrame) -> dict:
    pos = get_positions(df)
    dist = pairwise_distance_matrix(pos)
    np.fill_diagonal(dist, np.inf)

    nearest = dist.min(axis=1)

    return {
        "nn_mean": float(np.mean(nearest)),
        "nn_std": float(np.std(nearest)),
        "nn_min": float(np.min(nearest)),
        "nn_max": float(np.max(nearest)),
    }


def summarize_nearest_neighbors_by_layer(df_lower: pd.DataFrame, df_upper: pd.DataFrame) -> dict:
    lower_nn = summarize_nearest_neighbors(df_lower)
    upper_nn = summarize_nearest_neighbors(df_upper)

    return {
        "lower_nn_mean": lower_nn["nn_mean"],
        "lower_nn_std": lower_nn["nn_std"],
        "lower_nn_min": lower_nn["nn_min"],
        "lower_nn_max": lower_nn["nn_max"],
        "upper_nn_mean": upper_nn["nn_mean"],
        "upper_nn_std": upper_nn["nn_std"],
        "upper_nn_min": upper_nn["nn_min"],
        "upper_nn_max": upper_nn["nn_max"],
    }

def get_atom_type_counts(df: pd.DataFrame, prefix: str = "") -> dict:
    counts = df["symbol"].value_counts().to_dict()
    return {
        f"{prefix}count_C": int(counts.get("C", 0)),
        f"{prefix}count_H": int(counts.get("H", 0)),
        f"{prefix}count_O": int(counts.get("O", 0)),
    }

def build_structure_features(df: pd.DataFrame) -> dict:
    validate_layered_table(df)

    structure_id = df["structure_id"].iloc[0]
    relative_cif_path = df["relative_cif_path"].iloc[0]

    df_lower = df[df["layer_label"] == 0].copy()
    df_upper = df[df["layer_label"] == 1].copy()

    lower_centroid = compute_centroid(df_lower)
    upper_centroid = compute_centroid(df_upper)

    lower_com = compute_center_of_mass(df_lower)
    upper_com = compute_center_of_mass(df_upper)

    centroid_separation = float(np.linalg.norm(upper_centroid - lower_centroid))
    com_separation = float(np.linalg.norm(upper_com - lower_com))

    lower_intra = summarize_intralayer_distances(df_lower)
    upper_intra = summarize_intralayer_distances(df_upper)
    inter = summarize_interlayer_distances(df_lower, df_upper)

    nn_all = summarize_nearest_neighbors(df)
    nn_by_layer = summarize_nearest_neighbors_by_layer(df_lower, df_upper)

    features = {
        "structure_id": structure_id,
        "relative_cif_path": relative_cif_path,

        "atom_count": int(len(df)),
        "lower_atom_count": int(len(df_lower)),
        "upper_atom_count": int(len(df_upper)),

        **get_atom_type_counts(df, prefix=""),
        **get_atom_type_counts(df_lower, prefix="lower_"),
        **get_atom_type_counts(df_upper, prefix="upper_"),

        "lower_centroid_x": float(lower_centroid[0]),
        "lower_centroid_y": float(lower_centroid[1]),
        "lower_centroid_z": float(lower_centroid[2]),

        "upper_centroid_x": float(upper_centroid[0]),
        "upper_centroid_y": float(upper_centroid[1]),
        "upper_centroid_z": float(upper_centroid[2]),

        "lower_com_x": float(lower_com[0]),
        "lower_com_y": float(lower_com[1]),
        "lower_com_z": float(lower_com[2]),

        "upper_com_x": float(upper_com[0]),
        "upper_com_y": float(upper_com[1]),
        "upper_com_z": float(upper_com[2]),

        "centroid_separation": centroid_separation,
        "com_separation": com_separation,

        "lower_intralayer_dist_count": lower_intra["count"],
        "lower_intralayer_dist_mean": lower_intra["mean"],
        "lower_intralayer_dist_std": lower_intra["std"],
        "lower_intralayer_dist_min": lower_intra["min"],
        "lower_intralayer_dist_max": lower_intra["max"],

        "upper_intralayer_dist_count": upper_intra["count"],
        "upper_intralayer_dist_mean": upper_intra["mean"],
        "upper_intralayer_dist_std": upper_intra["std"],
        "upper_intralayer_dist_min": upper_intra["min"],
        "upper_intralayer_dist_max": upper_intra["max"],

        "interlayer_dist_count": inter["count"],
        "interlayer_dist_mean": inter["mean"],
        "interlayer_dist_std": inter["std"],
        "interlayer_dist_min": inter["min"],
        "interlayer_dist_max": inter["max"],

        "nn_mean": nn_all["nn_mean"],
        "nn_std": nn_all["nn_std"],
        "nn_min": nn_all["nn_min"],
        "nn_max": nn_all["nn_max"],

        **nn_by_layer,
    }

    return features

### Testing one structure

In [31]:
sample_path = atom_table_files[0]
df_sample = pd.read_csv(sample_path)

sample_features = build_structure_features(df_sample)

print("Sample file:", sample_path.name)
sample_features

Sample file: L0_D0_U0.csv


{'structure_id': 'L0_D0_U0',
 'relative_cif_path': 'r0/t0/t0_0.cif',
 'atom_count': 156,
 'lower_atom_count': 78,
 'upper_atom_count': 78,
 'count_C': 48,
 'count_H': 72,
 'count_O': 36,
 'lower_count_C': 24,
 'lower_count_H': 36,
 'lower_count_O': 18,
 'upper_count_C': 24,
 'upper_count_H': 36,
 'upper_count_O': 18,
 'lower_centroid_x': 9.39658634565746,
 'lower_centroid_y': 5.451754990029484,
 'lower_centroid_z': 4.891518548625557,
 'upper_centroid_x': 9.042279650625492,
 'upper_centroid_y': 4.412226421489943,
 'upper_centroid_z': 12.709464746262695,
 'lower_com_x': 9.269714296745065,
 'lower_com_y': 5.4767376062550905,
 'lower_com_z': 4.8164495250936525,
 'upper_com_x': 8.877279461019112,
 'upper_com_y': 4.420774300456403,
 'upper_com_z': 12.628835728338311,
 'centroid_separation': 7.894709344117953,
 'com_separation': 7.893189582933634,
 'lower_intralayer_dist_count': 3003,
 'lower_intralayer_dist_mean': 7.194368545764235,
 'lower_intralayer_dist_std': 3.86870338598656,
 'lower_int

### Parsing all files

In [32]:
feature_rows = []
failure_rows = []

for file_path in tqdm(atom_table_files, desc="Extracting structural features"):
    try:
        df = pd.read_csv(file_path)
        features = build_structure_features(df)
        feature_rows.append(features)

    except Exception as e:
        structure_id = None
        relative_cif_path = None

        try:
            if len(df) > 0:
                structure_id = df.get("structure_id", pd.Series([None])).iloc[0]
                relative_cif_path = df.get("relative_cif_path", pd.Series([None])).iloc[0]
        except Exception:
            pass

        failure_rows.append({
            "file_name": file_path.name,
            "structure_id": structure_id,
            "relative_cif_path": relative_cif_path,
            "error": f"{type(e).__name__}: {e}"
        })

Extracting structural features:   0%|          | 0/2916 [00:00<?, ?it/s]

### Saving outputs and validation

In [33]:
df_features = pd.DataFrame(feature_rows)
df_failures = pd.DataFrame(failure_rows)

df_features.to_csv(FEATURE_CSV, index=False)
df_failures.to_csv(FEATURE_FAILURES_CSV, index=False)

print("Saved:", FEATURE_CSV)
print("Saved:", FEATURE_FAILURES_CSV)

Saved: D:\masters_project\outputs\structural_features\structure_features.csv
Saved: D:\masters_project\outputs\structural_features\feature_failures.csv


In [34]:
print("=" * 60)
print("STRUCTURAL FEATURE EXTRACTION SUMMARY")
print("=" * 60)
print("Requested structures:", len(atom_table_files))
print("Processed successfully:", len(df_features))
print("Failures:", len(df_failures))

if len(df_features) > 0:
    display(df_features.head())

if len(df_failures) > 0:
    display(df_failures.head())

STRUCTURAL FEATURE EXTRACTION SUMMARY
Requested structures: 2916
Processed successfully: 2916
Failures: 0


,structure_id,relative_cif_path,atom_count,lower_atom_count,upper_atom_count,count_C,count_H,count_O,lower_count_C,lower_count_H,...,nn_min,nn_max,lower_nn_mean,lower_nn_std,lower_nn_min,lower_nn_max,upper_nn_mean,upper_nn_std,upper_nn_min,upper_nn_max
0,L0_D0_U0,r0/t0/t0_0.cif,156,78,78,48,72,36,24,36,...,0.977309,2.003689,1.134992,0.166349,0.977309,2.003689,1.118772,0.128105,0.977336,1.447277
1,L0_D0_U100,r0/t0/t0_100.cif,156,78,78,48,72,36,24,36,...,0.977156,1.993357,1.135027,0.165855,0.977488,1.993357,1.118810,0.127971,0.977156,1.446371
2,L0_D0_U120,r0/t0/t0_120.cif,156,78,78,48,72,36,24,36,...,0.977049,1.989993,1.135063,0.165559,0.977481,1.989993,1.118725,0.128255,0.977049,1.444283
3,L0_D0_U140,r0/t0/t0_140.cif,156,78,78,48,72,36,24,36,...,0.977482,2.013701,1.135251,0.167179,0.977482,2.013701,1.118680,0.128171,0.977948,1.444940
4,L0_D0_U160,r0/t0/t0_160.cif,156,78,78,48,72,36,24,36,...,0.977586,2.011914,1.135184,0.167162,0.977586,2.011914,1.118596,0.128099,0.977693,1.445462


# Task 7

### Path

In [49]:
GRAPH_DIR = OUTPUT_ROOT / "graphs"
GRAPH_OBJECT_DIR = GRAPH_DIR / "graph_objects"

GRAPH_METADATA_CSV = GRAPH_DIR / "graph_metadata.csv"
GRAPH_FAILURES_CSV = GRAPH_DIR / "graph_failures.csv"

MASTER_CSV = OUTPUT_ROOT / "dataset_master.csv"

DISTANCE_CUTOFF = 4.5
INCLUDE_SELF_LOOPS = False
UNDIRECTED_GRAPH = True

In [50]:
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_OBJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Layered atom tables:", LAYERED_ATOM_TABLE_DIR)
print("Graph output:", GRAPH_DIR)

Project root: D:\masters_project
Layered atom tables: D:\masters_project\outputs\layered_structures\atom_tables
Graph output: D:\masters_project\outputs\graphs


### Loader and Helpers

In [51]:
df_master = pd.read_csv(MASTER_CSV)

print("Master rows:", len(df_master))
df_master.head()

Master rows: 2916


,structure_id,relative_cif_path,lower_rotation,displacement,upper_rotation,energy,delta_energy
0,L0_D0_U0,r0/t0/t0_0.cif,0.0,0.0,0.0,-936.50959,0.13021
1,L0_D0_U20,r0/t0/t0_20.cif,0.0,0.0,20.0,-936.45119,0.18861
2,L0_D0_U40,r0/t0/t0_40.cif,0.0,0.0,40.0,-936.20394,0.43586
3,L0_D0_U60,r0/t0/t0_60.cif,0.0,0.0,60.0,-936.12172,0.51808
4,L0_D0_U80,r0/t0/t0_80.cif,0.0,0.0,80.0,-936.10592,0.53388


In [52]:
def validate_graph_input(df: pd.DataFrame) -> None:
    required_cols = {
        "atom_index",
        "atom_label",
        "symbol",
        "atomic_number",
        "x", "y", "z",
        "layer_label",
        "structure_id",
        "relative_cif_path",
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if len(df) == 0:
        raise ValueError("Empty atom table")

def get_positions(df: pd.DataFrame) -> np.ndarray:
    return df[["x", "y", "z"]].to_numpy(dtype=float)


def compute_distance_matrix(positions: np.ndarray) -> np.ndarray:
    diff = positions[:, None, :] - positions[None, :, :]
    dist = np.linalg.norm(diff, axis=2)
    return dist

def get_edge_type(layer_i: int, layer_j: int) -> int:
    """
    Edge type:
    0 = intralayer
    1 = interlayer
    """
    return 0 if layer_i == layer_j else 1

def build_edge_table(
    df_atoms: pd.DataFrame,
    distance_cutoff: float,
    include_self_loops: bool = False,
    undirected_graph: bool = True,
) -> pd.DataFrame:
    positions = get_positions(df_atoms)
    dist_matrix = compute_distance_matrix(positions)

    atom_indices = df_atoms["atom_index"].to_numpy(dtype=int)
    layer_labels = df_atoms["layer_label"].to_numpy(dtype=int)

    edge_rows = []
    n = len(df_atoms)

    for i in range(n):
        for j in range(n):
            if not include_self_loops and i == j:
                continue

            if undirected_graph and j <= i:
                continue

            distance = dist_matrix[i, j]

            if distance <= distance_cutoff:
                edge_type = get_edge_type(layer_labels[i], layer_labels[j])

                edge_rows.append({
                    "source_atom_index": int(atom_indices[i]),
                    "target_atom_index": int(atom_indices[j]),
                    "distance": float(distance),
                    "edge_type": int(edge_type),
                    "edge_type_name": "intralayer" if edge_type == 0 else "interlayer",
                    "source_layer": int(layer_labels[i]),
                    "target_layer": int(layer_labels[j]),
                })

                if undirected_graph:
                    edge_rows.append({
                        "source_atom_index": int(atom_indices[j]),
                        "target_atom_index": int(atom_indices[i]),
                        "distance": float(distance),
                        "edge_type": int(edge_type),
                        "edge_type_name": "intralayer" if edge_type == 0 else "interlayer",
                        "source_layer": int(layer_labels[j]),
                        "target_layer": int(layer_labels[i]),
                    })

    df_edges = pd.DataFrame(edge_rows)
    return df_edges

def build_node_table(df_atoms: pd.DataFrame) -> pd.DataFrame:
    node_cols = [
        "atom_index",
        "atom_label",
        "symbol",
        "atomic_number",
        "x", "y", "z",
        "frac_x", "frac_y", "frac_z",
        "layer_label",
        "layer_name",
        "structure_id",
        "relative_cif_path",
    ]

    available_cols = [c for c in node_cols if c in df_atoms.columns]
    return df_atoms[available_cols].copy()

def build_graph_metadata(
    df_nodes: pd.DataFrame,
    df_edges: pd.DataFrame,
    structure_id: str,
    relative_cif_path: str,
    distance_cutoff: float,
) -> dict:
    intralayer_edge_count = int((df_edges["edge_type"] == 0).sum()) if len(df_edges) > 0 else 0
    interlayer_edge_count = int((df_edges["edge_type"] == 1).sum()) if len(df_edges) > 0 else 0

    metadata = {
        "structure_id": structure_id,
        "relative_cif_path": relative_cif_path,
        "node_count": int(len(df_nodes)),
        "edge_count": int(len(df_edges)),
        "intralayer_edge_count": intralayer_edge_count,
        "interlayer_edge_count": interlayer_edge_count,
        "distance_cutoff": float(distance_cutoff),
        "has_edges": bool(len(df_edges) > 0),
    }

    if len(df_edges) > 0:
        metadata["edge_distance_mean"] = float(df_edges["distance"].mean())
        metadata["edge_distance_std"] = float(df_edges["distance"].std(ddof=0))
        metadata["edge_distance_min"] = float(df_edges["distance"].min())
        metadata["edge_distance_max"] = float(df_edges["distance"].max())
    else:
        metadata["edge_distance_mean"] = np.nan
        metadata["edge_distance_std"] = np.nan
        metadata["edge_distance_min"] = np.nan
        metadata["edge_distance_max"] = np.nan

    return metadata

def attach_structure_targets(metadata: dict, df_master: pd.DataFrame) -> dict:
    structure_id = metadata["structure_id"]

    row = df_master.loc[df_master["structure_id"] == structure_id]

    if len(row) == 0:
        metadata["energy"] = np.nan
        metadata["delta_energy"] = np.nan
        metadata["lower_rotation"] = np.nan
        metadata["displacement"] = np.nan
        metadata["upper_rotation"] = np.nan
        return metadata

    row = row.iloc[0]

    metadata["energy"] = float(row["energy"]) if pd.notna(row["energy"]) else np.nan
    metadata["delta_energy"] = float(row["delta_energy"]) if pd.notna(row["delta_energy"]) else np.nan
    metadata["lower_rotation"] = float(row["lower_rotation"]) if pd.notna(row["lower_rotation"]) else np.nan
    metadata["displacement"] = float(row["displacement"]) if pd.notna(row["displacement"]) else np.nan
    metadata["upper_rotation"] = float(row["upper_rotation"]) if pd.notna(row["upper_rotation"]) else np.nan

    return metadata

def build_graph_object(df_atoms: pd.DataFrame, df_master: pd.DataFrame, distance_cutoff: float) -> dict:
    validate_graph_input(df_atoms)

    structure_id = df_atoms["structure_id"].iloc[0]
    relative_cif_path = df_atoms["relative_cif_path"].iloc[0]

    df_nodes = build_node_table(df_atoms)
    df_edges = build_edge_table(
        df_atoms=df_atoms,
        distance_cutoff=distance_cutoff,
        include_self_loops=INCLUDE_SELF_LOOPS,
        undirected_graph=UNDIRECTED_GRAPH,
    )

    metadata = build_graph_metadata(
        df_nodes=df_nodes,
        df_edges=df_edges,
        structure_id=structure_id,
        relative_cif_path=relative_cif_path,
        distance_cutoff=distance_cutoff,
    )

    metadata = attach_structure_targets(metadata, df_master)

    graph = {
        "structure_id": structure_id,
        "relative_cif_path": relative_cif_path,
        "nodes": df_nodes,
        "edges": df_edges,
        "metadata": metadata,
    }

    return graph

### Testing one cif File

In [53]:
sample_path = atom_table_files[0]
df_sample = pd.read_csv(sample_path)

sample_graph = build_graph_object(
    df_atoms=df_sample,
    df_master=df_master,
    distance_cutoff=DISTANCE_CUTOFF,
)

print("Structure ID:", sample_graph["structure_id"])
print("Metadata:")
sample_graph["metadata"]

Structure ID: L0_D0_U0
Metadata:


{'structure_id': 'L0_D0_U0',
 'relative_cif_path': 'r0/t0/t0_0.cif',
 'node_count': 156,
 'edge_count': 3814,
 'intralayer_edge_count': 3520,
 'interlayer_edge_count': 294,
 'distance_cutoff': 4.5,
 'has_edges': True,
 'edge_distance_mean': 3.0955426268304613,
 'edge_distance_std': 0.9104130307759917,
 'edge_distance_min': 0.9773087022553408,
 'edge_distance_max': 4.498701638391007,
 'energy': -936.50959,
 'delta_energy': 0.1302100000000337,
 'lower_rotation': 0.0,
 'displacement': 0.0,
 'upper_rotation': 0.0}

In [54]:
display(sample_graph["nodes"].head())
display(sample_graph["edges"].head())

,atom_index,atom_label,symbol,atomic_number,x,y,z,frac_x,frac_y,frac_z,layer_label,layer_name,structure_id,relative_cif_path
0,86,H87,H,1,11.942418,5.677838,1.673773,0.622223,0.317492,0.059558,0,lower,L0_D0_U0,r0/t0/t0_0.cif
1,50,H51,H,1,2.593276,5.678713,1.674395,0.122246,0.317540,0.059580,0,lower,L0_D0_U0,r0/t0/t0_0.cif
2,54,H55,H,1,3.302254,7.994250,2.182526,0.152765,0.447070,0.077661,0,lower,L0_D0_U0,r0/t0/t0_0.cif
3,90,H91,H,1,12.652875,7.993063,2.185238,0.652836,0.447003,0.077757,0,lower,L0_D0_U0,r0/t0/t0_0.cif
4,53,H54,H,1,1.507792,8.102162,2.287225,0.056777,0.453083,0.081386,0,lower,L0_D0_U0,r0/t0/t0_0.cif


,source_atom_index,target_atom_index,distance,edge_type,edge_type_name,source_layer,target_layer
0,86,90,2.475199,0,intralayer,0,0
1,90,86,2.475199,0,intralayer,0,0
2,86,89,2.726092,0,intralayer,0,0
3,89,86,2.726092,0,intralayer,0,0
4,86,63,3.866345,0,intralayer,0,0


### All files

In [55]:
metadata_rows = []
failure_rows = []

for file_path in tqdm(atom_table_files, desc="Building graphs"):
    try:
        df_atoms = pd.read_csv(file_path)

        graph = build_graph_object(
            df_atoms=df_atoms,
            df_master=df_master,
            distance_cutoff=DISTANCE_CUTOFF,
        )

        structure_id = graph["structure_id"]

        save_path = GRAPH_OBJECT_DIR / f"{structure_id}.pkl"
        with open(save_path, "wb") as f:
            pickle.dump(graph, f)

        metadata = graph["metadata"].copy()
        metadata["graph_file"] = save_path.name
        metadata_rows.append(metadata)

    except Exception as e:
        structure_id = None
        relative_cif_path = None

        try:
            if len(df_atoms) > 0:
                structure_id = df_atoms.get("structure_id", pd.Series([None])).iloc[0]
                relative_cif_path = df_atoms.get("relative_cif_path", pd.Series([None])).iloc[0]
        except Exception:
            pass

        failure_rows.append({
            "file_name": file_path.name,
            "structure_id": structure_id,
            "relative_cif_path": relative_cif_path,
            "error": f"{type(e).__name__}: {e}"
        })

Building graphs:   0%|          | 0/2916 [00:00<?, ?it/s]

### Save Outputs and Validation

In [56]:
df_graph_metadata = pd.DataFrame(metadata_rows)
df_graph_failures = pd.DataFrame(failure_rows)

df_graph_metadata.to_csv(GRAPH_METADATA_CSV, index=False)
df_graph_failures.to_csv(GRAPH_FAILURES_CSV, index=False)

print("Saved:", GRAPH_METADATA_CSV)
print("Saved:", GRAPH_FAILURES_CSV)

Saved: D:\masters_project\outputs\graphs\graph_metadata.csv
Saved: D:\masters_project\outputs\graphs\graph_failures.csv


In [57]:
print("=" * 60)
print("GRAPH BUILD SUMMARY")
print("=" * 60)
print("Requested structures:", len(atom_table_files))
print("Built successfully:", len(df_graph_metadata))
print("Failures:", len(df_graph_failures))

if len(df_graph_metadata) > 0:
    display(df_graph_metadata.head())

if len(df_graph_failures) > 0:
    display(df_graph_failures.head())

GRAPH BUILD SUMMARY
Requested structures: 2916
Built successfully: 2916
Failures: 0


,structure_id,relative_cif_path,node_count,edge_count,intralayer_edge_count,interlayer_edge_count,distance_cutoff,has_edges,edge_distance_mean,edge_distance_std,edge_distance_min,edge_distance_max,energy,delta_energy,lower_rotation,displacement,upper_rotation,graph_file
0,L0_D0_U0,r0/t0/t0_0.cif,156,3814,3520,294,4.5,True,3.095543,0.910413,0.977309,4.498702,-936.50959,0.13021,0.0,0.0,0.0,L0_D0_U0.pkl
1,L0_D0_U100,r0/t0/t0_100.cif,156,3762,3504,258,4.5,True,3.086993,0.911601,0.977156,4.498597,-936.20154,0.43826,0.0,0.0,100.0,L0_D0_U100.pkl
2,L0_D0_U120,r0/t0/t0_120.cif,156,3768,3508,260,4.5,True,3.077230,0.902698,0.977049,4.498869,-936.24208,0.39772,0.0,0.0,120.0,L0_D0_U120.pkl
3,L0_D0_U140,r0/t0/t0_140.cif,156,3722,3510,212,4.5,True,3.079970,0.912671,0.977482,4.498572,-936.15696,0.48284,0.0,0.0,140.0,L0_D0_U140.pkl
4,L0_D0_U160,r0/t0/t0_160.cif,156,3652,3504,148,4.5,True,3.056393,0.906584,0.977586,4.498345,-936.07957,0.56023,0.0,0.0,160.0,L0_D0_U160.pkl


### Visuals one graph

In [58]:
example_graph_file = sorted(GRAPH_OBJECT_DIR.glob("*.pkl"))[0]

with open(example_graph_file, "rb") as f:
    example_graph = pickle.load(f)

print("Example graph file:", example_graph_file.name)
print("Graph keys:", list(example_graph.keys()))
print("Metadata:")
example_graph["metadata"]

Example graph file: L0_D0_U0.pkl
Graph keys: ['structure_id', 'relative_cif_path', 'nodes', 'edges', 'metadata']
Metadata:


{'structure_id': 'L0_D0_U0',
 'relative_cif_path': 'r0/t0/t0_0.cif',
 'node_count': 156,
 'edge_count': 3814,
 'intralayer_edge_count': 3520,
 'interlayer_edge_count': 294,
 'distance_cutoff': 4.5,
 'has_edges': True,
 'edge_distance_mean': 3.0955426268304613,
 'edge_distance_std': 0.9104130307759917,
 'edge_distance_min': 0.9773087022553408,
 'edge_distance_max': 4.498701638391007,
 'energy': -936.50959,
 'delta_energy': 0.1302100000000337,
 'lower_rotation': 0.0,
 'displacement': 0.0,
 'upper_rotation': 0.0}

# Task 8

### Path

In [ ]:
FEATURE_GRAPH_DIR = OUTPUT_ROOT / "featured_graphs"
FEATURE_GRAPH_OBJECT_DIR = FEATURE_GRAPH_DIR / "graph_objects"

FEATURE_GRAPH_METADATA_CSV = FEATURE_GRAPH_DIR / "featured_graph_metadata.csv"
FEATURE_GRAPH_FAILURES_CSV = FEATURE_GRAPH_DIR / "featured_graph_failures.csv"

In [60]:
FEATURE_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_GRAPH_OBJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input graph dir:", GRAPH_OBJECT_DIR)
print("Output feature graph dir:", FEATURE_GRAPH_DIR)

Project root: D:\masters_project
Input graph dir: D:\masters_project\outputs\graphs\graph_objects
Output feature graph dir: D:\masters_project\outputs\featured_graphs


In [61]:
graph_files = sorted(GRAPH_OBJECT_DIR.glob("*.pkl"))

if MAX_STRUCTURES is not None:
    graph_files = graph_files[:MAX_STRUCTURES]

Graph files found: 2916


[WindowsPath('D:/masters_project/outputs/graphs/graph_objects/L0_D0_U0.pkl'),
 WindowsPath('D:/masters_project/outputs/graphs/graph_objects/L0_D0_U100.pkl'),
 WindowsPath('D:/masters_project/outputs/graphs/graph_objects/L0_D0_U120.pkl'),
 WindowsPath('D:/masters_project/outputs/graphs/graph_objects/L0_D0_U140.pkl'),
 WindowsPath('D:/masters_project/outputs/graphs/graph_objects/L0_D0_U160.pkl')]

### Loader and Helpers

In [64]:
def validate_raw_graph(graph: dict) -> None:
    required_keys = {"structure_id", "nodes", "edges", "metadata"}
    missing = required_keys - set(graph.keys())

    if missing:
        raise ValueError(f"Missing graph keys: {sorted(missing)}")

    node_required = {"atom_index", "symbol", "x", "y", "z", "layer_label"}
    edge_required = {"source_atom_index", "target_atom_index", "distance", "edge_type"}

    node_missing = node_required - set(graph["nodes"].columns)
    edge_missing = edge_required - set(graph["edges"].columns)

    if node_missing:
        raise ValueError(f"Missing node columns: {sorted(node_missing)}")

    if len(graph["edges"]) > 0 and edge_missing:
        raise ValueError(f"Missing edge columns: {sorted(edge_missing)}")

def build_node_features(df_nodes: pd.DataFrame) -> np.ndarray:
    """
    Node features:
    [is_C, is_H, is_O, layer_label]
    """

    symbols = df_nodes["symbol"].astype(str).to_numpy()
    layer = df_nodes["layer_label"].to_numpy(dtype=float).reshape(-1, 1)

    is_C = (symbols == "C").astype(float).reshape(-1, 1)
    is_H = (symbols == "H").astype(float).reshape(-1, 1)
    is_O = (symbols == "O").astype(float).reshape(-1, 1)

    x = np.hstack([is_C, is_H, is_O, layer]).astype(np.float32)
    return x

def build_positions(df_nodes: pd.DataFrame) -> np.ndarray:
    pos = df_nodes[["x", "y", "z"]].to_numpy(dtype=np.float32)
    return pos

def build_edge_index(df_edges: pd.DataFrame) -> np.ndarray:
    if len(df_edges) == 0:
        return np.empty((2, 0), dtype=np.int64)

    edge_index = df_edges[["source_atom_index", "target_atom_index"]].to_numpy(dtype=np.int64).T
    return edge_index

def build_edge_index(df_edges: pd.DataFrame) -> np.ndarray:
    if len(df_edges) == 0:
        return np.empty((2, 0), dtype=np.int64)

    edge_index = df_edges[["source_atom_index", "target_atom_index"]].to_numpy(dtype=np.int64).T
    return edge_index

def build_edge_features(df_edges: pd.DataFrame) -> np.ndarray:
    """
    Edge features:
    [distance, is_intralayer, is_interlayer]
    """

    if len(df_edges) == 0:
        return np.empty((0, 3), dtype=np.float32)

    distance = df_edges["distance"].to_numpy(dtype=np.float32).reshape(-1, 1)
    edge_type = df_edges["edge_type"].to_numpy(dtype=int)

    is_intralayer = (edge_type == 0).astype(np.float32).reshape(-1, 1)
    is_interlayer = (edge_type == 1).astype(np.float32).reshape(-1, 1)

    edge_attr = np.hstack([distance, is_intralayer, is_interlayer]).astype(np.float32)
    return edge_attr
    
NODE_FEATURE_NAMES = [
    "is_C",
    "is_H",
    "is_O",
    "layer_label",
]

EDGE_FEATURE_NAMES = [
    "distance",
    "is_intralayer",
    "is_interlayer",
]

def build_featured_graph(raw_graph: dict) -> dict:
    validate_raw_graph(raw_graph)

    df_nodes = raw_graph["nodes"].copy()
    df_edges = raw_graph["edges"].copy()
    metadata = raw_graph["metadata"].copy()

    featured_graph = {
        "structure_id": raw_graph["structure_id"],
        "relative_cif_path": raw_graph.get("relative_cif_path"),
        "x": build_node_features(df_nodes),
        "pos": build_positions(df_nodes),
        "edge_index": build_edge_index(df_edges),
        "edge_attr": build_edge_features(df_edges),
        "node_feature_names": NODE_FEATURE_NAMES,
        "edge_feature_names": EDGE_FEATURE_NAMES,
        "metadata": metadata,
    }

    return featured_graph

### Testing one cif File

In [65]:
sample_graph_file = graph_files[0]

with open(sample_graph_file, "rb") as f:
    raw_graph = pickle.load(f)

featured_graph = build_featured_graph(raw_graph)

print("Structure ID:", featured_graph["structure_id"])
print("x shape:", featured_graph["x"].shape)
print("pos shape:", featured_graph["pos"].shape)
print("edge_index shape:", featured_graph["edge_index"].shape)
print("edge_attr shape:", featured_graph["edge_attr"].shape)

Structure ID: L0_D0_U0
x shape: (156, 4)
pos shape: (156, 3)
edge_index shape: (2, 3814)
edge_attr shape: (3814, 3)


In [66]:
print("Node feature names:", featured_graph["node_feature_names"])
print("Edge feature names:", featured_graph["edge_feature_names"])

print("\nFirst 5 node features:")
print(featured_graph["x"][:5])

print("\nFirst 5 positions:")
print(featured_graph["pos"][:5])

print("\nFirst 5 edge indices:")
print(featured_graph["edge_index"][:, :5])

print("\nFirst 5 edge features:")
print(featured_graph["edge_attr"][:5])

Node feature names: ['is_C', 'is_H', 'is_O', 'layer_label']
Edge feature names: ['distance', 'is_intralayer', 'is_interlayer']

First 5 node features:
[[0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]]

First 5 positions:
[[11.942417   5.677838   1.6737729]
 [ 2.5932763  5.6787124  1.6743951]
 [ 3.302254   7.9942503  2.182526 ]
 [12.652875   7.9930625  2.185238 ]
 [ 1.5077918  8.102161   2.287225 ]]

First 5 edge indices:
[[86 90 86 89 86]
 [90 86 89 86 63]]

First 5 edge features:
[[2.4751992 1.        0.       ]
 [2.4751992 1.        0.       ]
 [2.7260919 1.        0.       ]
 [2.7260919 1.        0.       ]
 [3.8663447 1.        0.       ]]


### All files

In [67]:
metadata_rows = []
failure_rows = []

for graph_file in tqdm(graph_files, desc="Building featured graphs"):
    try:
        with open(graph_file, "rb") as f:
            raw_graph = pickle.load(f)

        featured_graph = build_featured_graph(raw_graph)

        structure_id = featured_graph["structure_id"]
        save_path = FEATURE_GRAPH_OBJECT_DIR / f"{structure_id}.pkl"

        with open(save_path, "wb") as f:
            pickle.dump(featured_graph, f)

        row = featured_graph["metadata"].copy()
        row["graph_file"] = graph_file.name
        row["featured_graph_file"] = save_path.name
        row["node_feature_dim"] = int(featured_graph["x"].shape[1])
        row["edge_feature_dim"] = int(featured_graph["edge_attr"].shape[1])
        row["num_nodes"] = int(featured_graph["x"].shape[0])
        row["num_edges"] = int(featured_graph["edge_attr"].shape[0])

        metadata_rows.append(row)

    except Exception as e:
        structure_id = None
        try:
            structure_id = raw_graph.get("structure_id")
        except Exception:
            pass

        failure_rows.append({
            "graph_file": graph_file.name,
            "structure_id": structure_id,
            "error": f"{type(e).__name__}: {e}"
        })

Building featured graphs:   0%|          | 0/2916 [00:00<?, ?it/s]

### Save Outputs and Validation

In [68]:
df_featured_metadata = pd.DataFrame(metadata_rows)
df_featured_failures = pd.DataFrame(failure_rows)

df_featured_metadata.to_csv(FEATURE_GRAPH_METADATA_CSV, index=False)
df_featured_failures.to_csv(FEATURE_GRAPH_FAILURES_CSV, index=False)

print("Saved:", FEATURE_GRAPH_METADATA_CSV)
print("Saved:", FEATURE_GRAPH_FAILURES_CSV)

Saved: D:\masters_project\outputs\featured_graphs\featured_graph_metadata.csv
Saved: D:\masters_project\outputs\featured_graphs\featured_graph_failures.csv


In [69]:
print("=" * 60)
print("FEATURE GRAPH SUMMARY")
print("=" * 60)
print("Requested graphs:", len(graph_files))
print("Built successfully:", len(df_featured_metadata))
print("Failures:", len(df_featured_failures))

if len(df_featured_metadata) > 0:
    display(df_featured_metadata.head())

if len(df_featured_failures) > 0:
    display(df_featured_failures.head())

FEATURE GRAPH SUMMARY
Requested graphs: 2916
Built successfully: 2916
Failures: 0


,structure_id,relative_cif_path,node_count,edge_count,intralayer_edge_count,interlayer_edge_count,distance_cutoff,has_edges,edge_distance_mean,edge_distance_std,...,delta_energy,lower_rotation,displacement,upper_rotation,graph_file,featured_graph_file,node_feature_dim,edge_feature_dim,num_nodes,num_edges
0,L0_D0_U0,r0/t0/t0_0.cif,156,3814,3520,294,4.5,True,3.095543,0.910413,...,0.13021,0.0,0.0,0.0,L0_D0_U0.pkl,L0_D0_U0.pkl,4,3,156,3814
1,L0_D0_U100,r0/t0/t0_100.cif,156,3762,3504,258,4.5,True,3.086993,0.911601,...,0.43826,0.0,0.0,100.0,L0_D0_U100.pkl,L0_D0_U100.pkl,4,3,156,3762
2,L0_D0_U120,r0/t0/t0_120.cif,156,3768,3508,260,4.5,True,3.077230,0.902698,...,0.39772,0.0,0.0,120.0,L0_D0_U120.pkl,L0_D0_U120.pkl,4,3,156,3768
3,L0_D0_U140,r0/t0/t0_140.cif,156,3722,3510,212,4.5,True,3.079970,0.912671,...,0.48284,0.0,0.0,140.0,L0_D0_U140.pkl,L0_D0_U140.pkl,4,3,156,3722
4,L0_D0_U160,r0/t0/t0_160.cif,156,3652,3504,148,4.5,True,3.056393,0.906584,...,0.56023,0.0,0.0,160.0,L0_D0_U160.pkl,L0_D0_U160.pkl,4,3,156,3652


### Visuals of one featured graph

In [70]:
example_featured_file = sorted(FEATURE_GRAPH_OBJECT_DIR.glob("*.pkl"))[0]

with open(example_featured_file, "rb") as f:
    example_featured_graph = pickle.load(f)

print("Example featured graph file:", example_featured_file.name)
print("Keys:", list(example_featured_graph.keys()))
print("x shape:", example_featured_graph["x"].shape)
print("edge_index shape:", example_featured_graph["edge_index"].shape)
print("edge_attr shape:", example_featured_graph["edge_attr"].shape)

Example featured graph file: L0_D0_U0.pkl
Keys: ['structure_id', 'relative_cif_path', 'x', 'pos', 'edge_index', 'edge_attr', 'node_feature_names', 'edge_feature_names', 'metadata']
x shape: (156, 4)
edge_index shape: (2, 3814)
edge_attr shape: (3814, 3)


# Task 9

### Path

In [74]:
FEATURED_GRAPH_DIR = OUTPUT_ROOT / "featured_graphs"
FEATURED_GRAPH_OBJECT_DIR = FEATURED_GRAPH_DIR / "graph_objects"
PROCESSED_GRAPH_DIR = OUTPUT_ROOT / "processed_graphs"
PROCESSED_GRAPH_OBJECT_DIR = PROCESSED_GRAPH_DIR / "graph_objects"

DATASET_INDEX_CSV = PROCESSED_GRAPH_DIR / "dataset_index.csv"
PROCESSING_FAILURES_CSV = PROCESSED_GRAPH_DIR / "processing_failures.csv"
PACKED_DATASET_FILE = PROCESSED_GRAPH_DIR / "all_graphs.pkl"

SAVE_PACKED_DATASET = True

In [75]:
PROCESSED_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_GRAPH_OBJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input featured graphs:", FEATURED_GRAPH_OBJECT_DIR)
print("Processed graph output:", PROCESSED_GRAPH_DIR)

Project root: D:\masters_project
Input featured graphs: D:\masters_project\outputs\featured_graphs\graph_objects
Processed graph output: D:\masters_project\outputs\processed_graphs


In [77]:
featured_graph_files = sorted(FEATURED_GRAPH_OBJECT_DIR.glob("*.pkl"))

if MAX_STRUCTURES is not None:
    featured_graph_files = featured_graph_files[:MAX_STRUCTURES]
featured_graph_files = sorted(FEATURED_GRAPH_OBJECT_DIR.glob("*.pkl"))

print("Featured graph files found:", len(featured_graph_files))


Featured graph files found: 2916


### Loader and Helpers

In [78]:
def validate_featured_graph(graph: dict) -> None:
    required_keys = {
        "structure_id",
        "x",
        "pos",
        "edge_index",
        "edge_attr",
        "metadata",
    }

    missing = required_keys - set(graph.keys())
    if missing:
        raise ValueError(f"Missing graph keys: {sorted(missing)}")

    x = graph["x"]
    pos = graph["pos"]
    edge_index = graph["edge_index"]
    edge_attr = graph["edge_attr"]

    if not isinstance(x, np.ndarray):
        raise TypeError("x must be a numpy array")

    if not isinstance(pos, np.ndarray):
        raise TypeError("pos must be a numpy array")

    if not isinstance(edge_index, np.ndarray):
        raise TypeError("edge_index must be a numpy array")

    if not isinstance(edge_attr, np.ndarray):
        raise TypeError("edge_attr must be a numpy array")

    if x.ndim != 2:
        raise ValueError(f"x must be 2D, got shape {x.shape}")

    if pos.ndim != 2 or pos.shape[1] != 3:
        raise ValueError(f"pos must have shape [N, 3], got {pos.shape}")

    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError(f"edge_index must have shape [2, E], got {edge_index.shape}")

    if edge_attr.ndim != 2:
        raise ValueError(f"edge_attr must be 2D, got shape {edge_attr.shape}")

    if x.shape[0] != pos.shape[0]:
        raise ValueError("x and pos must have same number of nodes")

    if edge_index.shape[1] != edge_attr.shape[0]:
        raise ValueError("edge_index and edge_attr must have same number of edges")

def build_processed_graph(featured_graph: dict) -> dict:
    validate_featured_graph(featured_graph)

    metadata = featured_graph["metadata"].copy()

    processed_graph = {
        "structure_id": featured_graph["structure_id"],
        "relative_cif_path": featured_graph.get("relative_cif_path"),
        "x": featured_graph["x"].astype(np.float32),
        "pos": featured_graph["pos"].astype(np.float32),
        "edge_index": featured_graph["edge_index"].astype(np.int64),
        "edge_attr": featured_graph["edge_attr"].astype(np.float32),
        "node_feature_names": featured_graph.get("node_feature_names", []),
        "edge_feature_names": featured_graph.get("edge_feature_names", []),
        "targets": {
            "energy": metadata.get("energy", np.nan),
            "delta_energy": metadata.get("delta_energy", np.nan),
        },
        "metadata": metadata,
    }

    return processed_graph

def build_dataset_index_row(processed_graph: dict, file_name: str) -> dict:
    metadata = processed_graph["metadata"]

    row = {
        "structure_id": processed_graph["structure_id"],
        "relative_cif_path": processed_graph.get("relative_cif_path"),
        "graph_file": file_name,
        "num_nodes": int(processed_graph["x"].shape[0]),
        "num_edges": int(processed_graph["edge_attr"].shape[0]),
        "node_feature_dim": int(processed_graph["x"].shape[1]),
        "edge_feature_dim": int(processed_graph["edge_attr"].shape[1]),
        "energy": metadata.get("energy", np.nan),
        "delta_energy": metadata.get("delta_energy", np.nan),
        "lower_rotation": metadata.get("lower_rotation", np.nan),
        "displacement": metadata.get("displacement", np.nan),
        "upper_rotation": metadata.get("upper_rotation", np.nan),
    }

    return row

### Testing one cif File

In [79]:
sample_file = featured_graph_files[0]

with open(sample_file, "rb") as f:
    featured_graph = pickle.load(f)

processed_graph = build_processed_graph(featured_graph)

print("Structure ID:", processed_graph["structure_id"])
print("x shape:", processed_graph["x"].shape)
print("pos shape:", processed_graph["pos"].shape)
print("edge_index shape:", processed_graph["edge_index"].shape)
print("edge_attr shape:", processed_graph["edge_attr"].shape)
print("Targets:", processed_graph["targets"])

Structure ID: L0_D0_U0
x shape: (156, 4)
pos shape: (156, 3)
edge_index shape: (2, 3814)
edge_attr shape: (3814, 3)
Targets: {'energy': -936.50959, 'delta_energy': 0.1302100000000337}


### All files

In [80]:
dataset_rows = []
failure_rows = []
all_graphs = []

for graph_file in tqdm(featured_graph_files, desc="Saving processed graphs"):
    try:
        with open(graph_file, "rb") as f:
            featured_graph = pickle.load(f)

        processed_graph = build_processed_graph(featured_graph)

        structure_id = processed_graph["structure_id"]
        save_path = PROCESSED_GRAPH_OBJECT_DIR / f"{structure_id}.pkl"

        with open(save_path, "wb") as f:
            pickle.dump(processed_graph, f)

        dataset_rows.append(
            build_dataset_index_row(processed_graph, save_path.name)
        )

        if SAVE_PACKED_DATASET:
            all_graphs.append(processed_graph)

    except Exception as e:
        structure_id = None
        try:
            structure_id = featured_graph.get("structure_id")
        except Exception:
            pass

        failure_rows.append({
            "input_file": graph_file.name,
            "structure_id": structure_id,
            "error": f"{type(e).__name__}: {e}"
        })

Saving processed graphs:   0%|          | 0/2916 [00:00<?, ?it/s]

### Save Outputs and Validation

In [81]:
if SAVE_PACKED_DATASET:
    with open(PACKED_DATASET_FILE, "wb") as f:
        pickle.dump(all_graphs, f)

    print("Saved packed dataset:", PACKED_DATASET_FILE)
else:
    print("Packed dataset saving is disabled.")

df_dataset_index = pd.DataFrame(dataset_rows)
df_processing_failures = pd.DataFrame(failure_rows)

df_dataset_index.to_csv(DATASET_INDEX_CSV, index=False)
df_processing_failures.to_csv(PROCESSING_FAILURES_CSV, index=False)

print("Saved:", DATASET_INDEX_CSV)
print("Saved:", PROCESSING_FAILURES_CSV)

Saved packed dataset: D:\masters_project\outputs\processed_graphs\all_graphs.pkl
Saved: D:\masters_project\outputs\processed_graphs\dataset_index.csv
Saved: D:\masters_project\outputs\processed_graphs\processing_failures.csv


In [82]:
print("=" * 60)
print("PROCESSED GRAPH DATASET SUMMARY")
print("=" * 60)
print("Requested graphs:", len(featured_graph_files))
print("Saved successfully:", len(df_dataset_index))
print("Failures:", len(df_processing_failures))

if len(df_dataset_index) > 0:
    print()
    print("Unique node counts:", sorted(df_dataset_index["num_nodes"].dropna().unique().tolist()))
    print("Unique node feature dims:", sorted(df_dataset_index["node_feature_dim"].dropna().unique().tolist()))
    print("Unique edge feature dims:", sorted(df_dataset_index["edge_feature_dim"].dropna().unique().tolist()))

display(df_dataset_index.head())

if len(df_processing_failures) > 0:
    display(df_processing_failures.head())

PROCESSED GRAPH DATASET SUMMARY
Requested graphs: 2916
Saved successfully: 2916
Failures: 0

Unique node counts: [156]
Unique node feature dims: [4]
Unique edge feature dims: [3]


,structure_id,relative_cif_path,graph_file,num_nodes,num_edges,node_feature_dim,edge_feature_dim,energy,delta_energy,lower_rotation,displacement,upper_rotation
0,L0_D0_U0,r0/t0/t0_0.cif,L0_D0_U0.pkl,156,3814,4,3,-936.50959,0.13021,0.0,0.0,0.0
1,L0_D0_U100,r0/t0/t0_100.cif,L0_D0_U100.pkl,156,3762,4,3,-936.20154,0.43826,0.0,0.0,100.0
2,L0_D0_U120,r0/t0/t0_120.cif,L0_D0_U120.pkl,156,3768,4,3,-936.24208,0.39772,0.0,0.0,120.0
3,L0_D0_U140,r0/t0/t0_140.cif,L0_D0_U140.pkl,156,3722,4,3,-936.15696,0.48284,0.0,0.0,140.0
4,L0_D0_U160,r0/t0/t0_160.cif,L0_D0_U160.pkl,156,3652,4,3,-936.07957,0.56023,0.0,0.0,160.0


### Visuals one graph

In [83]:
example_processed_file = sorted(PROCESSED_GRAPH_OBJECT_DIR.glob("*.pkl"))[0]

with open(example_processed_file, "rb") as f:
    example_processed_graph = pickle.load(f)

print("Example file:", example_processed_file.name)
print("Keys:", list(example_processed_graph.keys()))
print("Targets:", example_processed_graph["targets"])
print("x shape:", example_processed_graph["x"].shape)
print("pos shape:", example_processed_graph["pos"].shape)
print("edge_index shape:", example_processed_graph["edge_index"].shape)
print("edge_attr shape:", example_processed_graph["edge_attr"].shape)

Example file: L0_D0_U0.pkl
Keys: ['structure_id', 'relative_cif_path', 'x', 'pos', 'edge_index', 'edge_attr', 'node_feature_names', 'edge_feature_names', 'targets', 'metadata']
Targets: {'energy': -936.50959, 'delta_energy': 0.1302100000000337}
x shape: (156, 4)
pos shape: (156, 3)
edge_index shape: (2, 3814)
edge_attr shape: (3814, 3)
